# 12 - Multi-PDF Q&A with Sources

## Purpose

This notebook creates a source-aware Q&A assistant for multiple uploaded PDF files.

The chatbot searches across all indexed PDFs, retrieves the most relevant chunks, sends them to GPT, and returns an answer with source metadata.

Each answer includes the PDF file name, page number, and chunk ID used as supporting context.

## Input

This notebook loads the existing multi-PDF Chroma index from:

```text
/tmp/chroma_multi_pdf_v1

In [0]:
%pip install --upgrade pypdf langchain langchain-community langchain-openai langchain-text-splitters chromadb tiktoken openai

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import getpass

openai_api_key = getpass.getpass("Enter your OpenAI API key: ")

if not openai_api_key:
    raise ValueError("OpenAI API key was not entered.")

os.environ["OPENAI_API_KEY"] = openai_api_key

print("OpenAI API key loaded for this session")

In [0]:
import os
import shutil

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import TokenTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

pdf_folder_path = "/Volumes/workspace/365pdf/365pdfupload/multi_pdf_docs"

files_info = dbutils.fs.ls(pdf_folder_path)

pdf_files = [
    file_info.path.replace("dbfs:", "")
    for file_info in files_info
    if file_info.path.lower().endswith(".pdf")
]

print("PDF files found:", len(pdf_files))

for pdf_file in pdf_files:
    print(pdf_file)

all_pdf_docs = []

for pdf_file in pdf_files:
    print("\nLoading PDF:")
    print(pdf_file)

    loader = PyPDFLoader(pdf_file)
    docs = loader.load()

    file_name = os.path.basename(pdf_file)

    for doc in docs:
        doc.metadata["source_file"] = file_name
        doc.metadata["source_path"] = pdf_file
        doc.metadata["page_number"] = doc.metadata.get("page")
        doc.metadata["document_type"] = "pdf"

    all_pdf_docs.extend(docs)

print("\nTotal page documents loaded:", len(all_pdf_docs))

token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=500,
    chunk_overlap=50
)

multi_pdf_token_docs = token_splitter.split_documents(all_pdf_docs)

for i, doc in enumerate(multi_pdf_token_docs):
    doc.metadata["chunk_id"] = i

print("Token chunks created:", len(multi_pdf_token_docs))

embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

multi_pdf_persist_directory = "/tmp/chroma_multi_pdf_v1"

if os.path.exists(multi_pdf_persist_directory):
    shutil.rmtree(multi_pdf_persist_directory)
    print("Removed existing Chroma directory")

multi_pdf_vectorstore = Chroma.from_documents(
    documents=multi_pdf_token_docs,
    embedding=embedding,
    persist_directory=multi_pdf_persist_directory
)

print("Multi-PDF Chroma vector database created")
print("Persist directory:", multi_pdf_persist_directory)

multi_pdf_retriever = multi_pdf_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 12,
        "lambda_mult": 0.7
    }
)

print("Multi-PDF retriever created")

In [0]:
print("Indexed chunks:", len(multi_pdf_token_docs))

test_docs = multi_pdf_retriever.invoke("What is Agentic RAG?")

print("Retrieved docs:", len(test_docs))

for i, doc in enumerate(test_docs, start=1):
    print(f"\nDocument {i}")
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page_number"))
    print("Chunk:", doc.metadata.get("chunk_id"))
    print(doc.page_content[:500])
    print("-" * 80)

In [0]:
def format_multi_pdf_context(docs):
    context_text = ""
    source_lines = []

    for i, doc in enumerate(docs, start=1):
        source_file = doc.metadata.get("source_file", "Unknown file")
        page_number = doc.metadata.get("page_number", "Unknown page")
        chunk_id = doc.metadata.get("chunk_id", "Unknown chunk")
        document_type = doc.metadata.get("document_type", "Unknown type")

        context_text += f"""
Document {i}
Source File: {source_file}
Page Number: {page_number}
Chunk ID: {chunk_id}
Document Type: {document_type}

Content:
{doc.page_content}

--------------------
"""

        source_lines.append(
            f"- Document {i}: {source_file}, page={page_number}, chunk_id={chunk_id}"
        )

    return context_text, source_lines

In [0]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chat = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

multi_pdf_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a helpful multi-document research assistant.

Answer the user's question using the retrieved context from the uploaded PDF files.

Rules:
- Base your answer on the retrieved context.
- You may synthesize across multiple retrieved documents.
- Do not invent facts that are not supported by the context.
- If the exact phrase is not present, but the concept is clearly discussed, explain the concept using the retrieved context.
- If the retrieved context is not sufficient, say what is missing.
- Keep the answer clear and beginner-friendly.
- Do not invent sources, page numbers, or document names.
"""
    ),
    (
        "human",
        """
Retrieved context from uploaded documents:

{context}

User question:

{question}

Write a helpful answer based on the retrieved context.
"""
    )
])

print("Improved multi-PDF prompt ready")

In [0]:
def ask_multi_pdf(question):
    docs = multi_pdf_retriever.invoke(question)

    if len(docs) == 0:
        return "I could not find relevant information in the uploaded documents."

    context_text, source_lines = format_multi_pdf_context(docs)

    messages = multi_pdf_prompt.invoke({
        "context": context_text,
        "question": question
    })

    answer = chat.invoke(messages)
    answer_text = output_parser.invoke(answer)

    final_response = f"""
Answer:
{answer_text}

Sources used:
{chr(10).join(source_lines)}
"""

    return final_response

In [0]:
print(ask_multi_pdf("What is Agentic RAG?"))

In [0]:
print(ask_multi_pdf("What are the challenges of question answering over rich documents?"))

In [0]:
print(ask_multi_pdf("Compare agentic RAG and multimodal RAG."))

In [0]:
print(ask_multi_pdf("What is multimodal retrieval?"))

In [0]:
print(ask_multi_pdf("What is the capital of Japan?"))

In [0]:
def debug_multi_pdf_retrieval(question):
    docs = multi_pdf_retriever.invoke(question)

    print("Question:", question)
    print("Retrieved docs:", len(docs))

    for i, doc in enumerate(docs, start=1):
        print(f"\nDocument {i}")
        print("Source file:", doc.metadata.get("source_file"))
        print("Page number:", doc.metadata.get("page_number"))
        print("Chunk ID:", doc.metadata.get("chunk_id"))
        print("Preview:")
        print(doc.page_content[:800])
        print("-" * 80)

In [0]:
debug_multi_pdf_retrieval("What is Agentic RAG?")